# TRACE-RAF Artifact and Claim Verification

This notebook verifies one completed publication-candidate ZIP or its Kaggle-extracted
directory. It contains no expected metric constants. Every comparison is recomputed
from archived predictions and checked against the manifest-backed tables generated by
the training notebook.

Run `01_event_timeraf_kaggle_pipeline.ipynb` first. Then attach its final output bundle.

## 1. Locate and authenticate the completed run

In [1]:
from io import BytesIO
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import sys
import zipfile

import numpy as np
import pandas as pd

# May point to either the publication ZIP or Kaggle's extracted run directory.
FINAL_RUN_SOURCE_OVERRIDE = None

def candidate_archives():
    if FINAL_RUN_SOURCE_OVERRIDE and Path(FINAL_RUN_SOURCE_OVERRIDE).is_file():
        yield Path(FINAL_RUN_SOURCE_OVERRIDE)
    cwd = Path.cwd().resolve()
    for search_root in (cwd, cwd.parent):
        yield from sorted(
            search_root.glob('event_timeraf_publication_candidate_*.zip'), reverse=True
        )
        yield search_root / 'event_timeraf_final_run.zip'
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        yield from sorted(kaggle_input.rglob('event_timeraf_publication_candidate_*.zip'), reverse=True)
        yield from sorted(kaggle_input.rglob('event_timeraf_final_run.zip'), reverse=True)

def archive_layout(path):
    try:
        with zipfile.ZipFile(path) as bundle:
            members = [PurePosixPath(name) for name in bundle.namelist()]
            marker = next(
                (item for item in members if item.parts[-3:] == ('outputs', 'logs', 'run_manifest.json')),
                None,
            )
            if marker is None:
                return None
            if any(item.is_absolute() or '..' in item.parts for item in members):
                raise RuntimeError(f'Unsafe archive paths: {path}')
            return PurePosixPath(*marker.parts[:-3])
    except (FileNotFoundError, zipfile.BadZipFile):
        return None

def candidate_run_directories():
    if FINAL_RUN_SOURCE_OVERRIDE and Path(FINAL_RUN_SOURCE_OVERRIDE).is_dir():
        yield Path(FINAL_RUN_SOURCE_OVERRIDE)
    seen = set()
    for search_root in (Path.cwd().resolve(), Path('/kaggle/input')):
        if not search_root.exists():
            continue
        for marker in search_root.rglob('run_manifest.json'):
            if marker.parent.name != 'logs' or marker.parent.parent.name != 'outputs':
                continue
            run_root = marker.parents[2].resolve()
            if run_root in seen:
                continue
            seen.add(run_root)
            if (
                (run_root / 'src' / 'event_timeraf' / 'config.py').exists()
                and (run_root / 'configs' / 'default.yaml').exists()
            ):
                yield run_root

FINAL_RUN_ZIP = None
FINAL_RUN_ROOT = None
PREFIX = None
for candidate in candidate_archives():
    layout = archive_layout(candidate)
    if layout is not None:
        FINAL_RUN_ZIP, PREFIX = candidate.resolve(), layout
        break
if FINAL_RUN_ZIP is None:
    FINAL_RUN_ROOT = next(candidate_run_directories(), None)
if FINAL_RUN_ZIP is None and FINAL_RUN_ROOT is None:
    raise FileNotFoundError(
        'Run notebook 01 to completion, then attach its publication-candidate ZIP '
        'or extracted output Dataset. Set FINAL_RUN_SOURCE_OVERRIDE if discovery fails.'
    )

def member(relative):
    return (PREFIX / PurePosixPath(relative)).as_posix()

def read_bytes(relative):
    if FINAL_RUN_ZIP is not None:
        with zipfile.ZipFile(FINAL_RUN_ZIP) as bundle:
            return bundle.read(member(relative))
    return (FINAL_RUN_ROOT / PurePosixPath(relative)).read_bytes()

def read_json(relative):
    return json.loads(read_bytes(relative).decode('utf-8'))

def read_csv(relative):
    return pd.read_csv(BytesIO(read_bytes(relative)))

def read_parquet(relative):
    return pd.read_parquet(BytesIO(read_bytes(relative)))

def read_npz(relative):
    return np.load(BytesIO(read_bytes(relative)))

manifest = read_json('outputs/logs/run_manifest.json')
integrity_rows = []
for relative, expected in manifest['artifacts'].items():
    payload = read_bytes(relative)
    integrity_rows.append({
        'artifact': relative,
        'bytes_match': len(payload) == expected['bytes'],
        'sha256_match': hashlib.sha256(payload).hexdigest() == expected['sha256'],
    })
integrity = pd.DataFrame(integrity_rows)
display(integrity.groupby(['bytes_match', 'sha256_match']).size().rename('artifact_count').to_frame())
assert integrity[['bytes_match', 'sha256_match']].all().all()
print({
    'source': str(FINAL_RUN_ZIP or FINAL_RUN_ROOT),
    'source_kind': 'zip' if FINAL_RUN_ZIP is not None else 'extracted_directory',
    'prefix': str(PREFIX) if PREFIX is not None else None,
    'run_id': manifest['run_id'],
})

,,artifact_count
bytes_match,sha256_match,
True,True,105


{'source': '/kaggle/input/datasets/sabbir234/firstrunv6/event_timeraf_20260821T143600090063Z', 'source_kind': 'extracted_directory', 'prefix': None, 'run_id': '20260821T143600090063Z'}


## 2. Load the archived implementation and result tables

In [2]:
extraction_root = Path('/kaggle/working/event_timeraf_verification_source') if Path('/kaggle/working').exists() else Path('verification_outputs/source')
if extraction_root.exists():
    shutil.rmtree(extraction_root)
extraction_root.mkdir(parents=True)
if FINAL_RUN_ZIP is not None:
    with zipfile.ZipFile(FINAL_RUN_ZIP) as bundle:
        for name in bundle.namelist():
            path = PurePosixPath(name)
            relative = path.parts[len(PREFIX.parts):]
            if relative and relative[0] in {'src', 'configs'}:
                bundle.extract(name, extraction_root)
    project_root = extraction_root.joinpath(*PREFIX.parts)
else:
    shutil.copytree(FINAL_RUN_ROOT / 'src', extraction_root / 'src')
    shutil.copytree(FINAL_RUN_ROOT / 'configs', extraction_root / 'configs')
    project_root = extraction_root
archived_src = (project_root / 'src').resolve()
for module_name in list(sys.modules):
    if module_name == 'event_timeraf' or module_name.startswith('event_timeraf.'):
        del sys.modules[module_name]
sys.path = [entry for entry in sys.path if Path(entry or '.').resolve() != archived_src]
sys.path.insert(0, str(archived_src))

from event_timeraf.config import load_config
from event_timeraf.evaluation import (
    diebold_mariano_hac, exceedance_metrics, holm_adjust_pvalues,
    horizon_skill_table, interval_metrics, log_scale_metrics, metric_values,
    paired_block_bootstrap_loss_difference,
    paired_masked_block_bootstrap_loss_difference, quantile_forecast_metrics,
    select_exceedance_decision_thresholds,
)
import event_timeraf.evaluation as archived_evaluation
evaluation_source = Path(archived_evaluation.__file__).resolve()
if not evaluation_source.is_relative_to(archived_src):
    raise ImportError(
        f'Verification imported {evaluation_source}, not archived source {archived_src}'
    )
print({'verified_implementation': str(evaluation_source)})

cfg = load_config(project_root / 'configs' / 'default.yaml', project_root)
predictions_long = read_parquet('outputs/predictions/predictions.parquet')
main_results = read_csv('outputs/tables/main_results.csv')
ablation_saved = read_csv('outputs/tables/ablation_results.csv')
exceedance_saved = read_csv('outputs/tables/aqi_exceedance_metrics.csv')
aqi_decision_selection_saved = read_csv('outputs/tables/aqi_decision_threshold_selection.csv')
aqi_calibrated_saved = read_csv('outputs/tables/aqi_calibrated_decision_metrics.csv')
split_protocol_saved = read_csv('outputs/tables/split_protocol.csv')
development_stress_saved = read_csv('outputs/tables/development_stress_period_metrics.csv')
horizon_skill_saved = read_csv('outputs/tables/horizon_skill_vs_climatology.csv')
log_saved = read_csv('outputs/tables/log_scale_metrics.csv')
interval_saved = read_csv('outputs/tables/tsfm_interval_metrics.csv')
quantile_saved = read_csv('outputs/tables/tsfm_quantile_calibration.csv')
probabilistic_saved = read_csv('outputs/tables/tsfm_probabilistic_metrics.csv')
site_level_saved = read_csv('outputs/tables/site_level_sensitivity.csv')
site_design_saved = read_csv('outputs/tables/site_level_design.csv')
site_selection_saved = read_csv('outputs/tables/site_selection_audit.csv')
stride_models_saved = read_csv('outputs/tables/kb_stride_model_sensitivity.csv')
trace_gate_saved = read_csv('outputs/tables/trace_raf_gate_selection.csv')
trace_design_saved = read_csv('outputs/tables/trace_raf_design.csv')
trace_oof_saved = read_csv('outputs/tables/trace_raf_oof_audit.csv')
trace_subset_saved = read_csv('outputs/tables/trace_raf_subset_inference.csv')

run_id_sets = [
    set(frame['run_id']) for frame in (
        predictions_long, main_results, ablation_saved, exceedance_saved,
        aqi_decision_selection_saved, aqi_calibrated_saved, split_protocol_saved,
        development_stress_saved,
        horizon_skill_saved, log_saved, interval_saved, quantile_saved, probabilistic_saved,
        site_level_saved, site_design_saved,
        site_selection_saved, stride_models_saved, trace_gate_saved, trace_design_saved,
        trace_oof_saved,
        trace_subset_saved,
    )
]
assert all(values == {manifest['run_id']} for values in run_id_sets)

{'verified_implementation': '/kaggle/working/event_timeraf_verification_source/src/event_timeraf/evaluation.py'}


## 3. Recompute every overall model metric

In [3]:
def model_arrays(model):
    frame = predictions_long.loc[predictions_long['model'] == model].sort_values(['window_id', 'horizon'])
    horizons = int(frame['horizon'].max())
    return (
        frame['actual'].to_numpy().reshape(-1, horizons),
        frame['prediction'].to_numpy().reshape(-1, horizons),
    )

array_cache = {model: model_arrays(model) for model in sorted(predictions_long['model'].unique())}
recomputed_rows = []
for model, (actual, predicted) in array_cache.items():
    recomputed_rows.append({'run_id': manifest['run_id'], 'model': model, **metric_values(actual, predicted)})
recomputed = pd.DataFrame(recomputed_rows)
saved = main_results[['model', 'mse', 'mae', 'rmse', 'mape', 'smape', 'r2']]
comparison = recomputed.merge(saved, on='model', suffixes=('_recomputed', '_saved'), validate='one_to_one')
for metric in ('mse', 'mae', 'rmse', 'mape', 'smape', 'r2'):
    comparison[f'{metric}_absolute_error'] = np.abs(
        comparison[f'{metric}_recomputed'] - comparison[f'{metric}_saved']
    )
metric_error_columns = [name for name in comparison if name.endswith('_absolute_error')]
display(comparison[['model', *metric_error_columns]].sort_values('mse_absolute_error', ascending=False))
assert comparison[metric_error_columns].to_numpy().max() < 1e-5
trace = read_npz('outputs/predictions/trace_raf_predictions.npz')
assert np.allclose(
    trace['M13_test'],
    trace['test_base'] + trace['test_gate'][:, None] * trace['event_test_correction'],
)
assert np.allclose(trace['M13_test'], array_cache['M13_trace_raf'][1])
assert np.allclose(trace['A03_test'], array_cache['A03_trace_raf_no_event'][1])
assert trace_oof_saved['embargo_passed'].all()
display(main_results.sort_values('mse'))

,model,mse_absolute_error,mae_absolute_error,rmse_absolute_error,mape_absolute_error,smape_absolute_error,r2_absolute_error
3,A03_trace_raf_no_event,7.105427e-15,0.000000e+00,0.0,0.0,7.105427e-15,0.000000e+00
11,C04_xgb_context_event,7.105427e-15,0.000000e+00,0.0,0.0,0.000000e+00,5.551115e-17
20,M06_cosine_retrieval,7.105427e-15,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00
1,A01_xgb_random_retrieval,0.000000e+00,0.000000e+00,0.0,0.0,7.105427e-15,0.000000e+00
4,B00_dlinear,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00
2,A02_xgb_matched_event_placebo,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00
5,B01_patchtst,0.000000e+00,8.881784e-16,0.0,0.0,0.000000e+00,5.551115e-17
6,B02_lstm,0.000000e+00,0.000000e+00,0.0,0.0,7.105427e-15,5.551115e-17
8,C01_ridge_context,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,5.551115e-17
7,C00_hour_month_climatology,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,3.122502e-17


,run_id,model,subset,event_availability_mode,n_origins,n_points,horizon,mse,mae,rmse,mape,smape,r2
10,20260821T143600090063Z,C01_ridge_context,all,retrospective_event_start,2460,59040,overall,39.144471,4.079278,6.256554,64.069777,38.226796,0.457050
13,20260821T143600090063Z,B02_lstm,all,retrospective_event_start,2460,59040,overall,39.978172,4.094546,6.322829,62.838283,37.479163,0.445487
22,20260821T143600090063Z,M13_trace_raf,all,retrospective_event_start,2460,59040,overall,40.062151,4.134315,6.329467,67.442003,38.230536,0.444322
23,20260821T143600090063Z,A03_trace_raf_no_event,all,retrospective_event_start,2460,59040,overall,40.062241,4.134615,6.329474,67.446401,38.232689,0.444320
11,20260821T143600090063Z,C05_lightgbm_context,all,retrospective_event_start,2460,59040,overall,40.088786,4.151403,6.331571,67.911049,38.390883,0.443952
15,20260821T143600090063Z,C06_context_ensemble,all,retrospective_event_start,2460,59040,overall,40.114545,4.155516,6.333604,68.313769,38.441203,0.443595
12,20260821T143600090063Z,B00_dlinear,all,retrospective_event_start,2460,59040,overall,40.304908,4.097433,6.348615,64.358204,37.837279,0.440955
14,20260821T143600090063Z,B01_patchtst,all,retrospective_event_start,2460,59040,overall,40.766248,4.111122,6.384845,63.204963,38.031695,0.434556
24,20260821T143600090063Z,A00_full_without_events,all,retrospective_event_start,2460,59040,overall,40.849100,4.150827,6.391330,66.750607,38.298035,0.433406
16,20260821T143600090063Z,M07_xgb_cosine,all,retrospective_event_start,2460,59040,overall,40.934500,4.156980,6.398008,66.886958,38.296143,0.432222


## 4. Recompute bootstrap intervals, DM tests, and Holm adjustment

In [4]:
comparisons = {
    'M04_minus_M03_weather_calendar': ('M04_xgb_context', 'M03_xgb_pm25'),
    'M07_minus_M04_cosine_retrieval': ('M07_xgb_cosine', 'M04_xgb_context'),
    'M08_minus_M07_event_conditioning': ('M08_event_timeraf_no_drift', 'M07_xgb_cosine'),
    'M09_minus_M08_drift_features': ('M09_event_timeraf_full', 'M08_event_timeraf_no_drift'),
    'M09_minus_A00_events': ('M09_event_timeraf_full', 'A00_full_without_events'),
    'M09_minus_M04_full': ('M09_event_timeraf_full', 'M04_xgb_context'),
    'A01_minus_M04_random_control': ('A01_xgb_random_retrieval', 'M04_xgb_context'),
    'M09_minus_A01_random_control': ('M09_event_timeraf_full', 'A01_xgb_random_retrieval'),
    'C04_minus_M04_raw_event_features': ('C04_xgb_context_event', 'M04_xgb_context'),
    'A02_minus_M04_matched_feature_count': ('A02_xgb_matched_event_placebo', 'M04_xgb_context'),
    'C04_minus_A02_event_signal': ('C04_xgb_context_event', 'A02_xgb_matched_event_placebo'),
    'C05_minus_M04_lightgbm': ('C05_lightgbm_context', 'M04_xgb_context'),
    'C06_minus_C05_context_ensemble': ('C06_context_ensemble', 'C05_lightgbm_context'),
    'M13_minus_C06_trace_residual_gate': ('M13_trace_raf', 'C06_context_ensemble'),
    'A03_minus_C06_no_event_residual_gate': ('A03_trace_raf_no_event', 'C06_context_ensemble'),
    'M13_minus_A03_trace_event_conditioning': ('M13_trace_raf', 'A03_trace_raf_no_event'),
    'M13_minus_C05_trace_vs_best_baseline': ('M13_trace_raf', 'C05_lightgbm_context'),
    'B00_minus_M04_dlinear': ('B00_dlinear', 'M04_xgb_context'),
    'B01_minus_M04_patchtst': ('B01_patchtst', 'M04_xgb_context'),
    'B02_minus_M04_lstm': ('B02_lstm', 'M04_xgb_context'),
    'M11_minus_M10_retrieval': ('M11_chronos_event_retrieval', 'M10_frozen_chronos'),
    'M11_minus_P00_climatology_placebo': ('M11_chronos_event_retrieval', 'P00_chronos_climatology_fusion'),
    'M11_minus_P01_persistence_placebo': ('M11_chronos_event_retrieval', 'P01_chronos_persistence_fusion'),
    'M12_minus_M04_drift_router': ('M12_validation_drift_router', 'M04_xgb_context'),
}
rows = []
for comparison_name, (model_a, model_b) in comparisons.items():
    actual, prediction_a = array_cache[model_a]
    actual_b, prediction_b = array_cache[model_b]
    assert np.array_equal(actual, actual_b)
    for metric in ('mse', 'mae'):
        interval = paired_block_bootstrap_loss_difference(
            actual, prediction_a, prediction_b, metric,
            cfg.evaluation.bootstrap_block_hours, cfg.evaluation.bootstrap_resamples, cfg.seed,
        )
        dm = diebold_mariano_hac(
            actual, prediction_a, prediction_b, metric, cfg.evaluation.dm_hac_lags
        )
        rows.append({'comparison': comparison_name, 'metric': metric, **interval, **dm})
ablation_recomputed = pd.DataFrame(rows)
for _, indices in ablation_recomputed.groupby('metric').groups.items():
    indices = list(indices)
    ablation_recomputed.loc[indices, 'bootstrap_p_value_holm'] = holm_adjust_pvalues(
        ablation_recomputed.loc[indices, 'bootstrap_p_value'].to_numpy()
    )
    ablation_recomputed.loc[indices, 'dm_p_value_holm'] = holm_adjust_pvalues(
        ablation_recomputed.loc[indices, 'dm_p_value'].to_numpy()
    )
checked_columns = [
    'difference', 'ci_low', 'ci_high', 'bootstrap_p_value', 'dm_statistic',
    'dm_p_value', 'bootstrap_p_value_holm', 'dm_p_value_holm',
]
ablation_check = ablation_recomputed.merge(
    ablation_saved, on=['comparison', 'metric'], suffixes=('_recomputed', '_saved'), validate='one_to_one'
)
errors = []
for column in checked_columns:
    error_name = f'{column}_absolute_error'
    ablation_check[error_name] = np.abs(
        ablation_check[f'{column}_recomputed'] - ablation_check[f'{column}_saved']
    )
    errors.append(error_name)
display(ablation_check[['comparison', 'metric', *errors]])
assert ablation_check[errors].to_numpy().max() < 1e-10

trace_masks = read_npz('outputs/predictions/trace_raf_subset_masks.npz')
trace_subset_comparisons = {
    'M13_minus_C06_trace_residual_gate': ('M13_trace_raf', 'C06_context_ensemble'),
    'M13_minus_A03_trace_event_conditioning': ('M13_trace_raf', 'A03_trace_raf_no_event'),
}
# Use the lossless TRACE-RAF arrays that Notebook 01 used to create the
# subset-inference table. The long Parquet table is verified separately.
trace_array_cache = {
    'M13_trace_raf': (trace['test_actual'], trace['M13_test']),
    'C06_context_ensemble': (trace['test_actual'], trace['test_base']),
    'A03_trace_raf_no_event': (trace['test_actual'], trace['A03_test']),
}
for model, (actual, predicted) in trace_array_cache.items():
    parquet_actual, parquet_predicted = array_cache[model]
    np.testing.assert_allclose(actual, parquet_actual, rtol=0.0, atol=1e-12)
    np.testing.assert_allclose(predicted, parquet_predicted, rtol=0.0, atol=1e-12)

trace_rows = []
for subset in ('event', 'event_context', 'drift', 'non_event'):
    mask = trace_masks[subset].astype(bool)
    if int(mask.sum()) < cfg.evaluation.minimum_subset_origins:
        continue
    for comparison_name, (model_a, model_b) in trace_subset_comparisons.items():
        actual, prediction_a = trace_array_cache[model_a]
        actual_b, prediction_b = trace_array_cache[model_b]
        np.testing.assert_array_equal(actual, actual_b)
        for metric in ('mse', 'mae'):
            interval = paired_masked_block_bootstrap_loss_difference(
                actual, prediction_a, prediction_b, mask, metric,
                cfg.evaluation.bootstrap_block_hours,
                cfg.evaluation.bootstrap_resamples, cfg.seed,
            )
            trace_rows.append({
                'subset': subset, 'comparison': comparison_name,
                'metric': metric, **interval,
            })
trace_subset_recomputed = pd.DataFrame(trace_rows)
for _, indices in trace_subset_recomputed.groupby('metric').groups.items():
    indices = list(indices)
    trace_subset_recomputed.loc[indices, 'bootstrap_p_value_holm'] = holm_adjust_pvalues(
        trace_subset_recomputed.loc[indices, 'bootstrap_p_value'].to_numpy()
    )
trace_keys = ['subset', 'comparison', 'metric']
recomputed_keys = set(map(tuple, trace_subset_recomputed[trace_keys].to_numpy()))
saved_keys = set(map(tuple, trace_subset_saved[trace_keys].to_numpy()))
if recomputed_keys != saved_keys:
    raise AssertionError({
        'missing_recomputed_trace_rows': sorted(saved_keys - recomputed_keys),
        'unexpected_recomputed_trace_rows': sorted(recomputed_keys - saved_keys),
    })
trace_check = trace_subset_recomputed.merge(
    trace_subset_saved,
    on=trace_keys, suffixes=('_recomputed', '_saved'),
    validate='one_to_one',
)
trace_errors = []
for column in ('difference', 'ci_low', 'ci_high', 'bootstrap_p_value', 'bootstrap_p_value_holm'):
    error_name = f'{column}_absolute_error'
    trace_check[error_name] = np.abs(
        trace_check[f'{column}_recomputed'] - trace_check[f'{column}_saved']
    )
    trace_errors.append(error_name)
display(trace_check[['subset', 'comparison', 'metric', *trace_errors]])
trace_error_values = trace_check[trace_errors].to_numpy(dtype=float)
if not np.isfinite(trace_error_values).all():
    raise AssertionError('TRACE-RAF verification produced a non-finite discrepancy.')
maximum_trace_error = float(trace_error_values.max(initial=0.0))
assert maximum_trace_error < 1e-10, (
    'TRACE-RAF subset-inference mismatch; maximum absolute error is '
    f'{maximum_trace_error:.3e}. Inspect the displayed keyed table.'
)
print({'trace_subset_inference': 'VERIFIED', 'maximum_absolute_error': maximum_trace_error})

,comparison,metric,difference_absolute_error,ci_low_absolute_error,ci_high_absolute_error,bootstrap_p_value_absolute_error,dm_statistic_absolute_error,dm_p_value_absolute_error,bootstrap_p_value_holm_absolute_error,dm_p_value_holm_absolute_error
0,M04_minus_M03_weather_calendar,mse,0.000000e+00,0.000000e+00,1.110223e-16,0.000000e+00,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00
1,M04_minus_M03_weather_calendar,mae,0.000000e+00,8.326673e-17,5.551115e-17,5.551115e-17,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00
2,M07_minus_M04_cosine_retrieval,mse,0.000000e+00,0.000000e+00,9.714451e-17,0.000000e+00,2.220446e-16,0.000000e+00,0.000000e+00,0.000000e+00
3,M07_minus_M04_cosine_retrieval,mae,8.326673e-17,0.000000e+00,2.081668e-17,1.214306e-17,4.440892e-16,8.933826e-17,5.551115e-17,8.326673e-17
4,M08_minus_M07_event_conditioning,mse,5.551115e-17,0.000000e+00,0.000000e+00,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00,0.000000e+00
5,M08_minus_M07_event_conditioning,mae,3.122502e-17,7.632783e-17,9.194034e-17,5.551115e-17,2.220446e-16,5.551115e-17,0.000000e+00,0.000000e+00
6,M09_minus_M08_drift_features,mse,2.775558e-17,0.000000e+00,0.000000e+00,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00,0.000000e+00
7,M09_minus_M08_drift_features,mae,1.040834e-17,3.295975e-17,1.387779e-17,5.551115e-17,0.000000e+00,8.326673e-17,0.000000e+00,0.000000e+00
8,M09_minus_A00_events,mse,5.551115e-17,5.551115e-17,0.000000e+00,5.551115e-17,1.110223e-16,0.000000e+00,0.000000e+00,0.000000e+00
9,M09_minus_A00_events,mae,5.160802e-17,7.632783e-17,7.285839e-17,0.000000e+00,2.775558e-17,0.000000e+00,0.000000e+00,0.000000e+00


,subset,comparison,metric,difference_absolute_error,ci_low_absolute_error,ci_high_absolute_error,bootstrap_p_value_absolute_error,bootstrap_p_value_holm_absolute_error
0,event_context,M13_minus_C06_trace_residual_gate,mse,5.551115e-17,8.326673e-17,0.000000e+00,5.551115e-17,0.000000e+00
1,event_context,M13_minus_C06_trace_residual_gate,mae,1.734723e-17,1.734723e-17,4.857226e-17,0.000000e+00,0.000000e+00
2,event_context,M13_minus_A03_trace_event_conditioning,mse,5.399327e-17,5.551115e-17,6.938894e-18,0.000000e+00,0.000000e+00
3,event_context,M13_minus_A03_trace_event_conditioning,mae,9.844556e-17,3.816392e-17,3.729655e-17,5.551115e-17,0.000000e+00
4,drift,M13_minus_C06_trace_residual_gate,mse,8.326673e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
5,drift,M13_minus_C06_trace_residual_gate,mae,6.245005e-17,4.163336e-17,3.469447e-17,5.551115e-17,0.000000e+00
6,drift,M13_minus_A03_trace_event_conditioning,mse,1.040834e-17,6.938894e-17,1.561251e-17,0.000000e+00,0.000000e+00
7,drift,M13_minus_A03_trace_event_conditioning,mae,2.547875e-17,7.849624e-17,6.028164e-17,0.000000e+00,0.000000e+00
8,non_event,M13_minus_C06_trace_residual_gate,mse,7.632783e-17,5.551115e-17,5.551115e-17,0.000000e+00,0.000000e+00
9,non_event,M13_minus_C06_trace_residual_gate,mae,1.734723e-17,5.551115e-17,3.729655e-17,2.428613e-17,5.551115e-17


{'trace_subset_inference': 'VERIFIED', 'maximum_absolute_error': 9.84455572616838e-17}


## 5. Recompute operational, log-scale, and probabilistic metrics

In [5]:
climatology = array_cache['C00_hour_month_climatology'][1]
validation_arrays = read_npz('outputs/predictions/validation_predictions.npz')
exceedance_frames, calibrated_frames, selection_frames = [], [], []
horizon_frames, log_frames = [], []
for model, (actual, predicted) in array_cache.items():
    exceedance_frames.append(exceedance_metrics(
        actual, predicted, cfg.evaluation.aqi_thresholds, model, manifest['run_id']
    ))
    if model in validation_arrays.files:
        selection = select_exceedance_decision_thresholds(
            validation_arrays['actual'], validation_arrays[model],
            cfg.evaluation.aqi_thresholds, model, manifest['run_id']
        )
        selection_frames.append(selection)
        decision_thresholds = dict(zip(
            selection['threshold_ug_m3'],
            selection['selected_decision_threshold_ug_m3'],
        ))
    else:
        decision_thresholds = {}
    calibrated_frames.append(exceedance_metrics(
        actual, predicted, cfg.evaluation.aqi_thresholds, model, manifest['run_id'],
        decision_thresholds=decision_thresholds,
    ))
    horizon_frames.append(horizon_skill_table(
        actual, predicted, climatology, model, manifest['run_id']
    ))
    log_frames.append(log_scale_metrics(actual, predicted, model, manifest['run_id']))
operational_recomputed = pd.concat(exceedance_frames, ignore_index=True)
decision_selection_recomputed = pd.concat(selection_frames, ignore_index=True)
calibrated_recomputed = pd.concat(calibrated_frames, ignore_index=True)
horizon_recomputed = pd.concat(horizon_frames, ignore_index=True)
log_recomputed = pd.concat(log_frames, ignore_index=True)

def maximum_table_error(left, right, keys, columns):
    merged = left.merge(right, on=keys, suffixes=('_recomputed', '_saved'), validate='one_to_one')
    errors = []
    for column in columns:
        difference = np.abs(merged[f'{column}_recomputed'] - merged[f'{column}_saved']).to_numpy()
        finite = difference[np.isfinite(difference)]
        errors.append(float(finite.max()) if len(finite) else 0.0)
    return max(errors)

window_metadata = read_parquet('data/processed/window_metadata.parquet')
validation_metadata = window_metadata.loc[window_metadata['split'] == 'validation'].copy()
if len(validation_metadata) != len(validation_arrays['actual']):
    raise AssertionError(
        'Validation metadata and archived validation predictions have different lengths.'
    )
validation_origin_time = pd.to_datetime(validation_metadata['origin_time'], utc=True)
january_2025_mask = (
    (validation_origin_time >= pd.Timestamp('2025-01-01', tz='UTC'))
    & (validation_origin_time < pd.Timestamp('2025-02-01', tz='UTC'))
)
january_reference_mse = metric_values(
    validation_arrays['actual'][january_2025_mask],
    validation_arrays['C00_hour_month_climatology'][january_2025_mask],
)['mse']
development_stress_recomputed = []
for model in development_stress_saved['model']:
    if model not in validation_arrays.files:
        raise KeyError(f'Missing archived validation predictions for {model}')
    values = metric_values(
        validation_arrays['actual'][january_2025_mask],
        validation_arrays[model][january_2025_mask],
    )
    development_stress_recomputed.append({
        'model': model, **values, 'climatology_mse': january_reference_mse,
        'skill_vs_climatology': 1.0 - values['mse'] / january_reference_mse,
    })
development_stress_recomputed = pd.DataFrame(development_stress_recomputed)
assert maximum_table_error(
    development_stress_recomputed, development_stress_saved, ['model'],
    ['mse', 'mae', 'rmse', 'r2', 'climatology_mse', 'skill_vs_climatology'],
) < 1e-10
assert development_stress_saved['evaluation_role'].eq(
    'development_stress_analysis'
).all()

assert maximum_table_error(
    operational_recomputed, exceedance_saved, ['model', 'threshold_ug_m3'],
    ['precision', 'recall', 'f1', 'critical_success_index', 'auroc', 'average_precision'],
) < 1e-10
assert maximum_table_error(
    decision_selection_recomputed, aqi_decision_selection_saved,
    ['model', 'threshold_ug_m3'],
    ['selected_decision_threshold_ug_m3', 'validation_precision',
     'validation_recall', 'validation_f1'],
) < 1e-10
assert maximum_table_error(
    calibrated_recomputed, aqi_calibrated_saved, ['model', 'threshold_ug_m3'],
    ['forecast_decision_threshold_ug_m3', 'precision', 'recall', 'f1',
     'critical_success_index', 'auroc', 'average_precision'],
) < 1e-10
assert maximum_table_error(
    horizon_recomputed, horizon_skill_saved, ['model', 'horizon'],
    ['model_mse', 'climatology_mse', 'skill_vs_climatology'],
) < 1e-10
assert maximum_table_error(
    log_recomputed, log_saved, ['model'], ['mse', 'mae', 'rmse', 'r2']
) < 1e-8  # Float Parquet/CSV round-trip tolerance; below reporting precision.

tsfm = read_npz('outputs/predictions/tsfm_predictions.npz')
levels = tuple(tsfm['quantile_levels'].tolist())
actual = (
    tsfm['test_actual']
    if 'test_actual' in tsfm.files
    else array_cache['M10_frozen_chronos'][0]
)
calibration_frames, probability_frames, interval_frames = [], [], []
for model, values in {
    'M10_frozen_chronos': tsfm['test_quantiles'],
    'M11_chronos_event_retrieval': tsfm['fused_test_quantiles'],
}.items():
    calibration, probability = quantile_forecast_metrics(
        actual, values, levels, model, manifest['run_id']
    )
    calibration_frames.append(calibration)
    probability_frames.append(probability)
    interval_frames.append(interval_metrics(
        actual, values[..., levels.index(0.1)], values[..., levels.index(0.9)],
        0.2, model, manifest['run_id'],
    ))
calibration_recomputed = pd.concat(calibration_frames, ignore_index=True)
probability_recomputed = pd.concat(probability_frames, ignore_index=True)
interval_recomputed = pd.concat(interval_frames, ignore_index=True)
assert maximum_table_error(
    calibration_recomputed, quantile_saved, ['model', 'quantile'],
    ['empirical_cdf', 'calibration_error', 'pinball_loss'],
) < 1e-10
assert maximum_table_error(
    probability_recomputed, probabilistic_saved, ['model'],
    ['crps_quantile_approximation', 'mean_absolute_calibration_error'],
) < 1e-10
assert maximum_table_error(
    interval_recomputed, interval_saved, ['model'],
    ['empirical_coverage', 'mean_width', 'winkler_interval_score'],
) < (1e-10 if 'test_actual' in tsfm.files else 1e-5)
display(probabilistic_saved)
display(exceedance_saved)

,run_id,model,quantile_grid,crps_quantile_approximation,mean_absolute_calibration_error,monotonicity_correction_fraction,n_points
0,20260821T143600090063Z,M10_frozen_chronos,"0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90",3.187069,0.017028,0.0,59040
1,20260821T143600090063Z,M11_chronos_event_retrieval,"0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90",4.119503,0.175772,0.0,59040


,run_id,model,threshold_ug_m3,forecast_decision_threshold_ug_m3,decision_rule,aggregation,n_origins,observed_exceedances,forecast_exceedances,tp,...,fn,tn,precision,recall,f1,critical_success_index,specificity,balanced_accuracy,auroc,average_precision
0,20260821T143600090063Z,C00_hour_month_climatology,35.4,35.4,physical_threshold,24h_forecast_mean,2460,37,0,0,...,37,2423,NaN,0.000000,NaN,0.000000,1.000000,0.500000,0.406727,0.021179
1,20260821T143600090063Z,C00_hour_month_climatology,55.4,55.4,physical_threshold,24h_forecast_mean,2460,0,0,0,...,0,2460,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
2,20260821T143600090063Z,C02_calendar_retrieval,35.4,35.4,physical_threshold,24h_forecast_mean,2460,37,1,0,...,37,2422,0.000000,0.000000,NaN,0.000000,0.999587,0.499794,0.888501,0.066807
3,20260821T143600090063Z,C02_calendar_retrieval,55.4,55.4,physical_threshold,24h_forecast_mean,2460,0,0,0,...,0,2460,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
4,20260821T143600090063Z,C03_event_conditioned_retrieval,35.4,35.4,physical_threshold,24h_forecast_mean,2460,37,1,0,...,37,2422,0.000000,0.000000,NaN,0.000000,0.999587,0.499794,0.889984,0.065520
5,20260821T143600090063Z,C03_event_conditioned_retrieval,55.4,55.4,physical_threshold,24h_forecast_mean,2460,0,0,0,...,0,2460,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
6,20260821T143600090063Z,M00_persistence,35.4,35.4,physical_threshold,24h_forecast_mean,2460,37,42,12,...,25,2393,0.285714,0.324324,0.303797,0.179104,0.987619,0.655971,0.961428,0.217159
7,20260821T143600090063Z,M00_persistence,55.4,55.4,physical_threshold,24h_forecast_mean,2460,0,11,0,...,0,2449,0.000000,NaN,NaN,0.000000,0.995528,NaN,NaN,NaN
8,20260821T143600090063Z,M01_daily_seasonal,35.4,35.4,physical_threshold,24h_forecast_mean,2460,37,45,0,...,37,2378,0.000000,0.000000,NaN,0.000000,0.981428,0.490714,0.887146,0.068318
9,20260821T143600090063Z,M01_daily_seasonal,55.4,55.4,physical_threshold,24h_forecast_mean,2460,0,0,0,...,0,2460,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN


## 6. Recompute the site-level sensitivity arm

In [6]:
site_arrays = read_npz('outputs/predictions/site_level_predictions.npz')
site_rows = []
for site_id in site_design_saved['site_id']:
    site_key = str(site_id).replace('-', '_')
    actual = site_arrays[f'{site_key}_actual']
    climatology = site_arrays[f'{site_key}_climatology']
    event_mask = site_arrays[f'{site_key}_event_mask'].astype(bool)
    for subset, mask in {
        'all': np.ones(len(actual), dtype=bool),
        'event': event_mask,
        'non_event': ~event_mask,
    }.items():
        eligible = int(mask.sum()) >= cfg.evaluation.minimum_subset_origins
        climatology_mse = metric_values(actual[mask], climatology[mask])['mse'] if eligible else np.nan
        for model, suffix in {
            'S_M04_xgb_context': 'M04',
            'S_M08_event_retrieval': 'M08',
        }.items():
            predicted = site_arrays[f'{site_key}_{suffix}']
            scores = metric_values(actual[mask], predicted[mask]) if eligible else {
                'mse': np.nan, 'mae': np.nan, 'rmse': np.nan, 'r2': np.nan
            }
            site_rows.append({
                'site_id': site_id, 'subset': subset, 'model': model,
                **scores,
                'skill_vs_site_climatology': (
                    1 - scores['mse'] / climatology_mse
                    if eligible and climatology_mse > 0 else np.nan
                ),
            })
site_recomputed = pd.DataFrame(site_rows)
assert maximum_table_error(
    site_recomputed, site_level_saved, ['site_id', 'subset', 'model'],
    ['mse', 'mae', 'rmse', 'r2', 'skill_vs_site_climatology'],
) < 1e-10
display(site_level_saved)

,run_id,site_id,subset,model,n_origins,eligible_for_metrics,mse,mae,rmse,mape,smape,r2,skill_vs_site_climatology
0,20260821T143600090063Z,06-037-1103,all,S_M04_xgb_context,1631,True,68.281399,5.169022,8.263256,237.693222,38.613427,0.245089,0.299828
1,20260821T143600090063Z,06-037-1103,all,S_M08_event_retrieval,1631,True,69.897995,5.253250,8.360502,233.714785,39.152311,0.227216,0.283251
2,20260821T143600090063Z,06-037-1103,event,S_M04_xgb_context,23,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260821T143600090063Z,06-037-1103,event,S_M08_event_retrieval,23,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20260821T143600090063Z,06-037-1103,non_event,S_M04_xgb_context,1608,True,68.752427,5.171510,8.291708,236.318210,38.098649,0.240740,0.298822
5,20260821T143600090063Z,06-037-1103,non_event,S_M08_event_retrieval,1608,True,70.288427,5.250248,8.383819,232.015857,38.605590,0.223777,0.283157
6,20260821T143600090063Z,06-037-4009,all,S_M04_xgb_context,1732,True,73.506320,5.278874,8.573583,62.898211,41.077644,0.229650,0.202386
7,20260821T143600090063Z,06-037-4009,all,S_M08_event_retrieval,1732,True,73.095470,5.225913,8.549589,62.116589,40.688261,0.233955,0.206844
8,20260821T143600090063Z,06-037-4009,event,S_M04_xgb_context,33,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,20260821T143600090063Z,06-037-4009,event,S_M08_event_retrieval,33,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Final reproducibility gate

In [7]:
required_tables = {
    'window_origin_attrition.csv', 'split_protocol.csv',
    'development_stress_period_metrics.csv',
    'aqi_decision_threshold_selection.csv', 'aqi_calibrated_decision_metrics.csv',
    'kb_stride_sensitivity.csv',
    'kb_stride_model_sensitivity.csv', 'event_weight_sensitivity.csv',
    'event_weight_model_sensitivity.csv', 'event_category_candidate_composition.csv',
    'validation_group_faithfulness.csv',
    'drift_leaf_occupancy.csv',
    'neural_baseline_training.csv', 'feature_count_control_design.csv',
    'tsfm_placebo_fusion_validation.csv',
    'tsfm_quantile_calibration.csv', 'tsfm_probabilistic_metrics.csv',
    'site_level_sensitivity.csv', 'site_level_design.csv',
    'site_selection_audit.csv',
    'trace_raf_design.csv', 'trace_raf_gate_selection.csv', 'trace_raf_oof_audit.csv',
    'trace_raf_subset_inference.csv',
}
required_figures = {
    'mae_by_horizon.png', 'mse_by_horizon.png', 'forecast_case.png',
    'forecast_event_case.png',
    'retrieval_diagnostics.png', 'drift_scores.png',
}
if FINAL_RUN_ZIP is not None:
    with zipfile.ZipFile(FINAL_RUN_ZIP) as bundle:
        names = {PurePosixPath(name).name for name in bundle.namelist()}
else:
    names = {path.name for path in FINAL_RUN_ROOT.rglob('*') if path.is_file()}
missing = sorted((required_tables | required_figures) - names)
assert not missing, f'Missing publication artifacts: {missing}'
options = manifest['run_options']
gate_rows = [
    {'gate': 'manifest hashes', 'passed': bool(integrity['sha256_match'].all())},
    {'gate': '1/6/24-hour KB sweep', 'passed': set(options.get('kb_stride_values', [])) == {1, 6, 24}},
    {'gate': 'complete learned stride sweep', 'passed': bool(
        options.get('stride_model_sweep_completed')
        and len(stride_models_saved) == 3 * len(options.get('kb_stride_values', []))
    )},
    {'gate': 'journal baselines', 'passed': bool(options.get('journal_baselines_completed'))},
    {'gate': 'learned event-weight sweep', 'passed': bool(options.get('event_weight_model_sweep_completed'))},
    {'gate': 'matched feature-count control', 'passed': bool(options.get('matched_feature_count_control'))},
    {'gate': 'three-monitor site arm', 'passed': bool(options.get('site_level_sensitivity_completed'))},
    {'gate': 'Holm-adjusted DM inference', 'passed': bool(options.get('holm_adjustment'))},
    {'gate': 'native Chronos quantile grid', 'passed': bool(
        options.get('chronos_native_quantile_grid')
        and min(options.get('probabilistic_quantiles', [0])) >= 0.1
        and max(options.get('probabilistic_quantiles', [1])) <= 0.9
    )},
    {'gate': 'TRACE-RAF OOF residual memory', 'passed': bool(
        options.get('trace_raf_completed')
        and options.get('trace_raf_oof_embargo_passed')
        and trace_oof_saved['embargo_passed'].all()
    )},
    {'gate': 'TRACE-RAF validation-only stride and gate selection', 'passed': bool(
        options.get('trace_raf_stride_selection_completed')
        and set(options.get('trace_raf_stride_values', [])) == {1, 6, 24}
        and (~trace_design_saved['test_used_for_selection'].astype(str).str.lower().eq('true')).all()
        and trace_design_saved['selected_gate_strength'].between(0, 1).all()
        and int(trace_design_saved['selected_residual_kb_stride_hours'].iloc[0])
            in {1, 6, 24}
    )},
    {'gate': 'selection-independent final holdout', 'passed': bool(
        options.get('explicit_final_holdout')
        and options.get('test_start_date') == cfg.forecast.test_start_date
        and split_protocol_saved.loc[split_protocol_saved['split'] == 'test', 'origin_start']
            .astype(str).str[:10].eq(cfg.forecast.test_start_date).all()
    )},
    {'gate': 'January 2025 development stress analysis', 'passed': bool(
        options.get('development_stress_period') == 'january_2025'
        and options.get('development_stress_role') == 'validation_only_not_final_holdout'
        and development_stress_saved['period'].eq('january_2025').all()
        and development_stress_saved['evaluation_role'].eq(
            'development_stress_analysis'
        ).all()
    )},
    {'gate': 'validation-calibrated AQI decisions', 'passed': bool(
        options.get('aqi_validation_calibration_completed')
        and len(aqi_decision_selection_saved) == len(decision_selection_recomputed)
        and set(aqi_decision_selection_saved['model'])
            == set(decision_selection_recomputed['model'])
    )},
    {'gate': 'publication artifacts complete', 'passed': bool(
        options.get('publication_artifacts_complete')
    )},
]
gates = pd.DataFrame(gate_rows)
display(gates)
assert gates['passed'].all()
print('Verification complete for run', manifest['run_id'])

,gate,passed
0,manifest hashes,True
1,1/6/24-hour KB sweep,True
2,complete learned stride sweep,True
3,journal baselines,True
4,learned event-weight sweep,True
5,matched feature-count control,True
6,three-monitor site arm,True
7,Holm-adjusted DM inference,True
8,native Chronos quantile grid,True
9,TRACE-RAF OOF residual memory,True


Verification complete for run 20260821T143600090063Z
